# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## The question

Power creep is defined as "the strengthening of [a] game and its pieces over time possibly to the point where new pieces invalidate older ones" [(Magruder, 2022)](https://journals.sagepub.com/doi/full/10.1177/15554120211050812#bibr20-15554120211050812). This occurs because developers want to keep their games fresh and exciting in order to keep players coming back [(Magic the Gathering on new cards sets)](https://magic.wizards.com/en/news/card-preview/fire-it-2019-06-21). However, too much power creep can make a game unrecognizeable. One card game that is perhaps infamous for power creep is Yu-Gi-Oh. One search on Reddit will provide many posts from both dedicated Yu-Gi-Oh subreddits and other card game subreddits discussing the power creep in Yu-Gi-Oh. Some posts are as old as five years, some as recent as two months ago, showing how pervasive and long-lasting the issue of power creep is.

Yu-Gi-Oh has been around for nearly 30 years, and though power creep is an issue, it has still been going strong. Another franchise that has been going arguably stronger for the same length of time is Pokemon. It is so beloved that parents that grew up with Pokemon are now sharing it with their own children [(Amenabar, 2022)](https://www.washingtonpost.com/video-games/2022/08/10/pokemon-starter-parents-kids/). Yu-Gi-Oh has not experienced the same rite-of-passage experience as Pokemon has, in part due to how complicated the game is [(short Reddit thread about the subject)](https://www.reddit.com/r/Yugioh101/comments/onn3vm/dear_parents_do_your_kids_play_yugioh_and_is_it/). If Pokemon experiences power creep in the same way, this parental bonding method might not be as effective any more. Additionally, fans could get tired of having to buy all of the new games and DLC just to get access to all of the new, strong Pokemon. However, this would only occur if Pokemon experiences a high level of power creep.

Question: Has Pokemon experienced power creep over its thirty years of existance? If so, to what extent is power creep experienced?


## Where is the data coming from to answer this question?

All data collected to answer this question comes from [PokeAPI](https://pokeapi.co/docs/v2). This is a robust API offering endpoints for many different kinds of Pokemon data, but the three I will be collecting data on are their [Generation API](https://pokeapi.co/docs/v2#games-section), their [Pokemon Species API](https://pokeapi.co/docs/v2#pokemon-species), and their [Pokemon API](https://pokeapi.co/docs/v2#pokemon).

Suppelementary data will be retrieved from either [Bulbapedia](https://bulbapedia.bulbagarden.net/wiki/Main_Page), [Pokemon Database](https://pokemondb.net/), or [Serebii.net](https://www.serebii.net/).

## How will power creep be measured?

There are many ways to measure power creep in the complex game of Pokemon: the functionality of newly-introduced abilities compared to older abilities, the power and function of newly-introduced moves compared to older moves, and a Pokemon's move pool (the moves a Pokemon can learn) to name a few. My chosen unit of measurement for power creep, however, is the average Base Stat Total (BST) of Pokemon across the nine generations. I have chosen this measure because though BSTs do not represent the actual stats of a Pokemon in game, they represent the potential of how powerful a Pokemon could possibly be. Unchanged by outside factors, like abilities or type match-ups, the Pokemon with the higher stats is almost certainly going to win the battle. Additionally, as a purely quantitative measure, it easy to compare BSTs and see at a glance if Pokemon have overall been getting stronger through generations.

However, because Pokemon is such a complex game, solely looking at BSTs might not give a complete picture of power creep. Take my earlier statement that the Pokemon with higher stats is almost certainly going to win the battle. Included in that statement is a caveat about outside factors affecting the outcome. In Pokemon, outside factors will always be present and they will influence battles, so higher BST Pokemon are not always going to win. Though power creep can be present in these outside factors, it is harder to quanitfy in an easily understandable numerical format. Quantifying power creep in these factors is certainly possible, but it will require a lot more analysis and be subject to more bias depending on how someone chooses to quantify power creep. Using BST makes for a relatively simple, unbiased measure.

This exploration does not aim to be exhaustive; rather, this is a simple overview of power creep through a change in BSTs overtime. One is welcome to take the foundations laid in this analysis and perform their own deeper, more exhaustive analysis.



## Getting a list of all Pokemon

First, I am going to use PokeAPI's Generation API to collect a list of all of the Pokemon species introduced in every generation. A species, as defined by PokeAPI, "forms the basis for at least one Pokemon." For instance, take the Pokemon Wormadam. Wormadam has three forms based on its cloak: Plant Cloak, Sandy Cloak, and Trash Cloak, but Wormadam is the underlying species of all three forms.

In [2]:
#Initializing empty lists for storage
api_calls = []
mon_name = []
gen_list = []
gen_mon = {}

#Calling the Generation API
for i in range(1,10):
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='w') as f:
        request = requests.get(f"https://pokeapi.co/api/v2/generation/{i}")
        pokemon_json = request.json()
        json.dump(pokemon_json,f,indent=4)
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path= 'pokemon_species',
            meta = 'name',
            record_prefix= 'pokemon_'
        )

#Storing the API call URL and the Pokemon names for use in later API calls
    for i in range(len(df['pokemon_url'])):
        api_calls.append(df['pokemon_url'][i])
        mon_name.append(df['pokemon_name'][i])

#Creating a generation list for later usage in renaming columns
    gen_list.append(df['name'][0])

#Creating a dictionary of every Pokemon added in each Generation for later usage in assigning Generation to a dataframe
    for name in df['pokemon_name']:
        if df['name'][0] not in gen_mon.keys():
            gen_mon[df['name'][0]] = [name]
        else:
            gen_mon[df['name'][0]].append(name)


Now that I have a list of all the different Pokemon species' APIs, I am going to call each species' API to get the API calls for each Pokemon form individually (ex: all three types of Wormadam cloaks rather than just the Wormadam species as a whole). But, first, I am going to create a function to perform the remaining API calls for me.

### Pokemon Species Data

In [3]:
def call_api(call_list,name_list,file_suffix):
##Takes in a list of APIs to call, a list of names for the files, and a suffix for the file to call and store .json data from an API##
    for i in range(len(call_list)):
        with open(f'../data/raw/{name_list[i]}_{file_suffix}.json', mode='w') as f:
            request = requests.get(call_list[i])
            pokemon_json = request.json()
            json.dump(pokemon_json,f,indent=4)

In [4]:
call_api(api_calls,mon_name,'spec')

In [5]:
#Initializing empty lists for storage
stat_calls = []
form_name = []

#Getting the list of each Pokemon form from API alongside calls
for i in range(len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                sep = '_'
                )

        for i in range(len(df['pokemon_url'])):
                stat_calls.append(df['pokemon_url'][i])
                form_name.append(df['pokemon_name'][i])


### Pokemon Stats Data

In [6]:
call_api(stat_calls,form_name,'stat')

All of the necessary calls have been made, and all the data stored in the 'raw' folder under 'data'.

## Note: the raw data files are not included in the hand-in because they exceeded the size of Nuvolos's allowed hand-in file size

## Storing the collected data in dataframes

Now that the data has been collected and safely stored, it is time to arrange the data into neat dataframes.

### How many dataframes will be created?
There are going to be two different dataframes: one containing the species data of the Pokemon, and one containing the actual stats data of the Pokemon.

### What will each dataframe contain?
For the dataframe with the species data, each row is going to contain the ID of the Pokemon, the name of the Pokemon, the flags of whether the Pokemon is a baby, a legendary, or a mythical, and the Generation that the Pokemon is associated with.

For the dataframe with the stats data, each row is going to contain the ID, name of the Pokemon, and its associated Generation once again, and all of their individual base stats and the names of those stats.

### Why two different dataframes?

There are a couple of reasons why two different dataframes are needed:
- Having the species and the individual stats data separated out will make for easier analysis later, as I can choose to combine the dataframes with pd.merge() if the need arises, or I can keep them separate and perform individual analyses on the data they contain
- The dataframe containing the stats data has six rows for each Pokemon (one for each kind of base stat), while the dataframe containing the species data has one row for each Pokemon, so attempting to combine them out of the gate will lead to misaligned dataframes and bad data
- The data is stored in two different API calls, so creating two different dataframes is easier for me, personally

### Why am I creating and storing the dataframes here? Why not in NB02?

I initially wanted to create and store the dataframes in NB02, but I realized when trying to move the code over that the lists where all of my API calls are stored do not carry over between notebooks. As such, to avoid repeating code and having to initialize the same lists again in another place, I am going to create the dataframes here.

In [8]:
#Creating the dataframe with species data
spec_df = ''
with open(f'../data/raw/{mon_name[0]}_spec.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path = 'varieties',
            meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical'],
            sep='_'
        )
        gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})

initial_df = pd.concat([df,gen_df],axis=1)


for i in range(1, len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                        data,
                        record_path = 'varieties',
                        meta = ['id', 'name', 'is_baby', 'is_legendary', 'is_mythical'],
                        sep='_'
                )
                gen_df = pd.DataFrame(data['generation'],index=[0]).rename(columns={'name':'gen', 'url':'gen_url'})
        dfc = pd.concat([df,gen_df],axis=1)
        spec_df = pd.concat([initial_df,dfc],axis=0, ignore_index = True)
        initial_df = spec_df

The use of .rename(): When I was creating the dataframe, I found that I had to add Generation separately due to the way it was stored in the API and then concatenate it onto the dataframe by row, but when I tried to concatenate them originally, I ran into name errors since the name of the Generation and the URL to call the generation were under the same name as the column storing the Pokemon name and the URL to get data about the Pokemon themselves. As such, I used .rename() to change the conflicting names of columns to something else, passing a dictionary with the original column names as keys and my new column names as values.

In [9]:
## Creating the dataframe with stats data
with open(f'../data/raw/{form_name[0]}_stat.json', mode='r') as f:
    data = json.load(f)
    initial_df = pd.json_normalize(
        data,
        record_path = 'stats',
        meta = ['id', 'name'],
        sep='_'
    )

for i in range(1, len(stat_calls)):
    with open(f'../data/raw/{form_name[i]}_stat.json', mode='r') as f:
            data = json.load(f)
            df = pd.json_normalize(
                data,
                record_path = 'stats',
                meta = ['id','name'],
                sep='_'
            )
    stats_df = pd.concat([initial_df,df],axis=0, ignore_index = True)
    initial_df = stats_df

Now that the two dataframes have been created, it is time to save them as csv files for usage in later notebooks.

In [10]:
spec_df.to_csv('../data/processed/spec_df.csv', index=False)

stats_df.to_csv('../data/processed/stats_df.csv', index=False)